In [ ]:
import math
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from wandb.apis.public.runs import Run

import wandb

In [ ]:
api = wandb.Api()

In [ ]:
runs = api.runs("graphcore/llama-mobile", filters={"config.name": "lr-sweep-v2"})

In [ ]:
def cleanup_run(run: Run) -> dict[str, Any]:
    d = {}

    d["batch_size"] = run.config["training"]["batch_size"]
    d["n_steps"] = run.config["training"]["n_steps"]
    d["beta1"] = run.config["training"]["optimiser"]["betas"][0]
    d["beta2"] = run.config["training"]["optimiser"]["betas"][1]
    d["weight_decay"] = run.config["training"]["optimiser"]["weight_decay"]
    d["n_warmup_steps"] = run.config["training"]["lr_schedule"]["n_warmup_steps"]
    d["lr_exponent"] = math.log2(run.config["training"]["optimiser"]["lr"])

    history = run.history()
    d["train_loss"] = np.mean(history["train/loss"].tail(10).values)
    d["val_loss"] = run.summary["val/loss"]
    d["step_time"] = np.mean(history["perf/step_time"].values)

    return d

In [ ]:
df = pd.DataFrame([cleanup_run(run) for run in runs])
df.sort_values(by="val_loss", ascending=True, inplace=True)

In [ ]:
# NOTE: 2**-13 learning rate explodes, so exclude
df

In [ ]:
for bs, group in df[df["val_loss"] < 1].groupby("batch_size"):
    group_sorted = group.sort_values('lr_exponent')
    plt.plot(group_sorted["lr_exponent"], group_sorted["val_loss"], label=f"bs = {bs}")
    plt.yscale("log")
    
plt.title("Learning rate sweep")
plt.xlabel("Learning rate exponent")
plt.ylabel("KL divergence")
plt.legend(title="Batch size")
plt.grid(True)
plt.tight_layout()

In [ ]:
baseline = df[df["lr_exponent"] == -16]

In [ ]:
baseline

In [ ]:
# Additional hyperparameter sweeps
runs = api.runs("graphcore/llama-mobile", filters={"config.name": "lr-sweep-v2-hparams"})

In [ ]:
df = pd.DataFrame([cleanup_run(run) for run in runs])
df = pd.concat([baseline, df], ignore_index=True)

In [ ]:
df

In [ ]:
def plot_param_sweep(
    df: pd.DataFrame, baseline_config: dict[str, Any], param_name: str
) -> None:
    config = {k: v for k, v in baseline_config.items() if k != param_name}
    sweep_df = df.loc[(df[list(config)] == pd.Series(config)).all(axis=1)].sort_values(by=param_name, ascending=True)
    plt.plot(sweep_df[param_name], sweep_df["val_loss"])
    plt.xlabel(param_name)
    plt.ylabel("Validation loss")
    plt.title(param_name)

In [ ]:
baseline_config = {"beta2": 0.999, "weight_decay": 0.0, "n_warmup_steps": 0}


In [ ]:
# Plot beta2
plot_param_sweep(df, baseline_config, "beta2")

In [ ]:
# Plot weight decay
plot_param_sweep(df, baseline_config, "weight_decay")

In [ ]:
# Plot warmup
plot_param_sweep(df, baseline_config, "n_warmup_steps")